In [1]:
import re
import pandas as pd
import numpy as np
#from spyctral import core
from astropy.table import QTable
import astropy.units as u
import dateutil.parser

In [2]:
path = r'ruta_a_carpeta_st/case_SC_Starlight.out'

In [22]:
SL_GET_HEADER =  re.compile(r'\[')
SL_GET_TITLE_VALUE = re.compile(r"\[")
SL_REPLACE_AND = re.compile(r"&")
SL_GET_MULTIPLE_VALUES = re.compile(r',')
SL_GET_DATE = re.compile(r'\b\d{2}/[a-zA-Z]{3}/\d{4}\b')
PATRON = re.compile(r'#(.*?)(?:(?=\n)|$)')
    
def _proces_header(header_ln):
    """Recives as input a list with the lines of the header and returns a
    dictionary with the parameters and values"""

    head_dict = {}
    for sl in header_ln:
        if re.findall(SL_GET_DATE, sl):
            head_dict["Date"] = dateutil.parser.parse(
                re.findall(SL_GET_DATE, sl)[0]
            )
            head_dict["user"] = sl.split("[")[1].split("-")[0].strip()

        sl = re.sub(
            r"\s{2,}", " ", sl.replace("&", ",").replace("]\n", "")
        ).replace("/", "_")

        # Handles string with multiple values on the same line.
        # They are idenfied by havin a ','
        if re.findall(SL_GET_MULTIPLE_VALUES, sl):
            starline_values = (
                sl.split("[")[0].strip().split(" ")
            )  # keeps the values
            starline_var = sl.split("[")[1].split(
                ","
            )  # keeps the names of the variables

            for pos, val in enumerate(starline_values):
                if bool(
                    re.search(r"[0-9]", val)
                ):  # filters for numeric values
                    head_dict[starline_var[pos].strip().split(" ")[0]] = float(
                        val
                    )
                else:
                    head_dict[starline_var[pos].strip().split(" ")[0]] = val

        # Handles string with S/N titles that repeats
        # overlaps if not handled.
        elif re.findall(r"\[S_N", sl):
            starline_list = sl.split("[")
            head_dict[
                starline_list[1].replace(" ", "_").replace(".", "").strip()
            ] = float(starline_list[0])

        # saves all the other values
        # that not contain any special exception
        else:
            starline_list = sl.replace("-", "_").replace("#", "").split("[")
            if bool(re.search(r"[0-9]", starline_list[0])) and not bool(
                re.search(r"[a-z]", starline_list[0])
            ):  # filters for numeric values
                head_dict[starline_list[1].split(" ")[0]] = float(
                    starline_list[0].replace("_", "-").strip()
                )
            else:
                head_dict[starline_list[1].split(" ")[0]] = starline_list[
                    0
                ].strip()

    return head_dict


def _proces_tables(block_lines):
    """Recives a list that contains the lines of the tables and
    return a dictionary with 4 tables as values"""
    block_titles = []
    blocks = []
    tab = []
    for sl in block_lines:
        # This part procces the tables
        if (
            re.search(SL_GET_TITLE_VALUE, sl) is None
            and re.match("##", sl) is None
        ):
            # Filters the titles of the tables
            if re.search(r"# ", sl):
                block_titles.append(sl)
            # Filters the empty spaces
            elif len(sl) > 1:
                # print(sl)
                sl = re.sub(r"\s{2,}", " ", sl.strip())
                tab.append(sl.split(" "))

            # Generates a block for each empty line that founds
            # and if the block is larger it appends it
            else:
                if len(tab) > 1:
                    blocks.append(tab)
                    tab = []
    blocks.append(tab)
    tab = []
    first_title = (
        re.sub(r"\s{2,}", " ", block_titles[0][1:])
        .replace(".", "")
        .replace("?", "")
        .strip()
    )

    # Verificar que los elementos sean números
    for i, row in enumerate(blocks[4]):
        converted_row = []
        for item in row:
            try:
                # Intentar convertir el elemento en un número
                converted_item = float(item)
            except ValueError:
                # Si no se puede convertir a número, levantar una excepción
                raise ValueError(
                    f"Element at position ({i})"
                    " in 'synthetic_spectrum' table cannot "
                    "  be converted to a number."
                )
            converted_row.append(converted_item)
        blocks[4][i] = converted_row

    # Crear la tabla synthetic_spectrum con unidades
    synthetic_spectrum_table = QTable(
        rows=blocks[4], names=["l_obs", "f_obs", "f_syn", "weights"]
    )

    # Cambio: Agregar unidades a la columna 'l_obs'
    synthetic_spectrum_table["l_obs"].unit = u.AA

    spectra_dict = {
        # Cambio: Usar la tabla con unidades
        "synthetic_spectrum": synthetic_spectrum_table,
        "synthetic_results": QTable(
            rows=blocks[0], names=first_title.split(" ")
        ),
        "results_average_chains_xj": QTable(rows=blocks[1]),
        "results_average_chains_mj": QTable(rows=blocks[2]),
        "results_average_chains_Av_chi2_mass": QTable(
            rows=np.array(blocks[3]).T[1:],
            names=np.array(blocks[3]).T[0],
        ),
    }

    return spectra_dict


def read_starlight(path):
    """Recives as input a path from the location of the starlight file and
    returns a two dicctionaries the first is the header information and the
    second is the tables information"""

    header_lines, block_lines = [], []
    with open(path) as starfile:
        for d, starline in enumerate(starfile):
            if re.findall(SL_GET_TITLE_VALUE, starline):
                header_lines.append(starline)
            else:
                block_lines.append(starline)

    header_info = _proces_header(header_lines)

    tables_dict = _proces_tables(block_lines)

    # return header_info, tables_dict
    return header_info, tables_dict

In [23]:

dic1, dic2 = read_starlight(path)


In [51]:
dic2['synthetic_results']['x_j(%)'] = dic2['synthetic_results']['x_j(%)'].astype(float)
dic2['synthetic_results']['age_j(yr)']= dic2['synthetic_results']['age_j(yr)'].astype(float)
dic2['synthetic_results']["Z_j"]= dic2['synthetic_results']["Z_j"].astype(float)
sresults = dic2['synthetic_results']

In [52]:
df = sresults[sresults["x_j(%)"] >= 4]


In [53]:
edad = int(
            10
            ** (
                ((df["x_j(%)"] * np.log10(df["age_j(yr)"]))).sum()
                / df["x_j(%)"].sum()
            )
        )
edad = f"{edad:.3e}"  
print(edad)

5.602e+08


In [54]:
met = ((df["x_j(%)"] * df["Z_j"])).sum() / df["x_j(%)"].sum()
met = round(met, 5)  # redondeo la metalicidad a 5 decimales
print(met)

0.006


In [55]:
edad_1 = int(
    10
    ** (
        ((df["x_j(%)"] * np.log10(df["age_j(yr)"]))).sum()
        / df["x_j(%)"].sum()
    )
)
edad_1 = np.log10(edad_1)
edad_1 = round(edad_1, 2)
edad_1 = f"{edad_1:.2e}"
print(edad_1)

8.75e+00


In [57]:
import plotly.graph_objects as go
import plotly.offline as pyo

In [72]:
x = np.log10(df["age_j(yr)"])
np.around(x.value,2)

array([6.94, 7.74, 9.11, 9.11])

In [82]:
dic1['v0_min']

164.96

In [83]:
nombre_archivo = dic1['Cid@UFSC']
fig = go.Figure()
# ------------------------------------------------------------------------------------------------------------------------------
# ACA HAGO UN GRAFICO DE BURBUJA DEL APORTE DE CADA UNA DE LAS SSP Y DEL ESPECTRO SINTETICO RESULTANTE
# ------------------------------------------------------------------------------------------------------------------------------
x = np.log10(df["age_j(yr)"])
x = np.around(x.value,2)
y = df["Z_j"]
aporte = df["x_j(%)"]
porcen = [
    f"{valor:.1f}%" for valor in aporte
]  # esto es para que sobre la burjuja aparesca el aporte en forma de porcentaje

fig.add_scatter(
    x=x,
    y=y,
    mode="markers",
    marker=dict(size=30 + aporte, color="blue"),
    name="Aporte de SSP",
)  # SSP
fig.add_scatter(
    x=[edad],
    y=[met],
    mode="markers",
    marker=dict(size=20, color="red"),
    name="Promedio",
)  # Sintetico
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="text",
        text=porcen,
        textposition="middle center",
        name=" ",
    )
)  # texto sobre burbujas
# ------------------------------------------------------------------------------------------------------------------------------
# ACA AÑADO UNA NOTACION AL GRAFICO DE LA EDAD,METALICIDA Y CHI2 DEL ESPECTRO RESULTANTE
# ------------------------------------------------------------------------------------------------------------------------------
fig.add_annotation(
    xref="paper",
    yref="paper",
    x=1,
    y=1,
    text="Espectro sintetico: "
    + "Log(Edad) = "
    + f"{edad}"
    + " yr"
    + ", "
    + "Z = "
    + f"{met}"
    + ", "
    + "\u03C7\u00B2 = "
    + f"{round(float(dic1['Burn-In']))}"   
    + ", "
    + "Av = "
    + f"{round(float(dic1['AV_min']))}"
    + ", "
    + "V0 = "
    + f"{round(float(dic1['v0_min']),2)}"
    + " Km/s"
    + ", "
    + "Vd = "
    + f"{round(float(dic1['vd_min']),2)}"
    + " Km/s",
    showarrow=False,
    font=dict(size=18),
)

fig.update_layout(
    title="Grafico de burbuja del archivo " + f"{nombre_archivo}",
    title_font=dict(size=22),
)
fig.update_xaxes(title_text="Edad (yr)", title_font=dict(size=18))
fig.update_yaxes(title_text="Abundacia", title_font=dict(size=18))


pyo.plot(fig)  # si pongo fig.show() me lo muestra en terminal


'temp-plot.html'

In [87]:
df2= dic2['synthetic_spectrum']
nombre_archivo = dic1['Cid@UFSC']
fig = go.Figure()
# -------------------------------------------------------------------------------------------------------------------------------
# ACA GRAFICO EL ESPECTRO OBSERVADO, EL SINTETICO Y LA DIFERENCIA ENTRE AMBOS
# -------------------------------------------------------------------------------------------------------------------------------
fig.add_scatter(x=df2["l_obs"], y=df2["f_obs"], name="observado")
fig.add_scatter(x=df2["l_obs"], y=df2["f_syn"], name="modelo")
fig.add_scatter(
    x=df2["l_obs"],
    y=(df2["f_obs"] - df2["f_syn"]),
    name="residuos",
    line=dict(width=0.7),
)
# -------------------------------------------------------------------------------------------------------------------------------
# ACA CALCULO LA DESVIACION ESTANDAR DE LA DIFERENCIA ENTRE EL MODELO Y LOS DATOS
# -------------------------------------------------------------------------------------------------------------------------------
sigma = np.std(df2["f_obs"] - df2["f_syn"])
# -------------------------------------------------------------------------------------------------------------------------------
# ACA GRAFICO LAS RECTAS DEL RESIDUO
# -------------------------------------------------------------------------------------------------------------------------------
fig.add_scatter(
    x=df2["l_obs"],
    y=[0] * len(df2["l_obs"]),
    name="residuo 0",
    line=dict(color="black", width=1, dash="dash"),
)
# fig.add_scatter( x = df2['l_obs'], y = [3*sigma] * len(df2['l_obs']), name = '+3 sigmas', line = dict(color = 'black',width = 1,dash = 'dash'))
# fig.add_scatter( x = df2['l_obs'], y = [-3*sigma] * len(df2['l_obs']), name ='-3 sigmas', line = dict(color = 'black',width = 1,dash = 'dash'))
# -------------------------------------------------------------------------------------------------------------------------------
# ACA AÑADO LAS ANOTACIONES AL GRAFICO
# -------------------------------------------------------------------------------------------------------------------------------
fig.add_annotation(
    xref="paper",
    yref="paper",
    x=1,
    y=1,
    text="Espectro sintetico: "
    + "Log(Edad) = "
    + f"{edad}"
    + " yr"
    + ", "
    + "Z = "
    + f"{met}"
    + ", "
    + "\u03C7\u00B2 = "
    + f"{round(float(dic1['Burn-In']))}"   
    + ", "
    + "Av = "
    + f"{round(float(dic1['AV_min']))}"
    + ", "
    + "V0 = "
    + f"{round(float(dic1['v0_min']),2)}"
    + " Km/s"
    + ", "
    + "Vd = "
    + f"{round(float(dic1['vd_min']),2)}"
    + " Km/s",
    showarrow=False,
    font=dict(size=18),
)


# -------------------------------------------------------------------------------------------------------------------------------
# ACA CONFIGURO LOS DATALLES DEL GRAFICO
# -------------------------------------------------------------------------------------------------------------------------------
fig.update_layout(
    title="Resuldatos de StarLight del archivo " + f"{nombre_archivo}",
    title_font=dict(size=22),
)
# fig.update_layout(legend=dict(x=0, y=1, bgcolor='rgba(255, 255, 255, 0.5)', bordercolor='gray', borderwidth=1))
fig.update_xaxes(title_text="Lambda", title_font=dict(size=18))
fig.update_yaxes(title_text="Flujo", title_font=dict(size=18))

# fig.show()
pyo.plot(fig)  # si pongo fig.show() me lo muestra en terminal

'temp-plot.html'